# Suspicious outlet-photo detection — GPU run

Runs the `outlet-audit` pipeline on a GPU (Tesla T4 or similar).

**Steps**

1. **Check the GPU** — `nvidia-smi` must list a device. No device = runtime has no GPU; switch runtime to GPU first.
2. **Get the code + dataset** — zip the repo locally (`zip -r Suspicious_Photo_Detection.zip Suspicious_Photo_Detection -x 'Suspicious_Photo_Detection/.venv/*' 'Suspicious_Photo_Detection/cache/*' 'Suspicious_Photo_Detection/.git/*' 'Suspicious_Photo_Detection/results/*'`), upload it to the root of Google Drive. The cell mounts Drive, unzips to `/content/Suspicious_Photo_Detection` and `%cd`s into it (`%cd` persists across cells, `!cd` does not).
3. **Install the package** — `pip install -e .` installs `outlet-audit` from this repo. The torch check must print `True`; if it prints `False`, the install pulled a CPU-only torch. Fix with:
   `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128`
4. **Run the pipeline** — `--device cuda` moves DINOv2, CLIP and EasyOCR to the GPU. `--batch-size 64` is safe on a 16 GB T4. First run downloads model weights (~1 GB). The SIFT/RANSAC geometry stage is CPU-only and unaffected by the GPU.

Outputs land in `results/`: per-outlet JSON/CSV plus an HTML report (`--report`). Expects `dataset/<outlet_id>/*.jpg`.


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/IM-Assesment.zip -d /content/repo
import glob, os
%cd {os.path.dirname(glob.glob('/content/repo/**/pyproject.toml', recursive=True)[0])}
!ls

In [ ]:
!pip install -q -e .
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
!outlet-audit run --data dataset --out results --config config.yaml --device cuda --batch-size 64 --report